In [2]:
import numpy as np
import pandas as pd

# Параметры сетки
n = 4  # количество шагов по x (точек: 0, 0.25, 0.5, 0.75, 1.0)
h = 1.0 / n
x_vals = np.linspace(0, 1, n + 1)
y_vals = np.linspace(0, 1, n + 1)

# Внутренние узлы: (n-1)^2 = 9 переменных
N = (n - 1)**2
A = np.zeros((N, N))
b = np.zeros(N)

# Граничные условия
def left(y): return -y**2 + 1
def right(y): return y
def bottom(x): return (np.sin(x) - (1 + np.sin(1))) * (x**3 + 1)
def top(x): return x

# Нумерация узлов: (i, j) → индекс в x = (i - 1) * (n - 1) + (j - 1)
def idx(i, j):
    return (i - 1) * (n - 1) + (j - 1)

# Заполнение матрицы A и правой части b
for i in range(1, n):
    for j in range(1, n):
        k = idx(i, j)
        A[k, k] = -4

        if i > 1:
            A[k, idx(i - 1, j)] = 1
        else:
            b[k] -= left(y_vals[j])

        if i < n - 1:
            A[k, idx(i + 1, j)] = 1
        else:
            b[k] -= right(y_vals[j])

        if j > 1:
            A[k, idx(i, j - 1)] = 1
        else:
            b[k] -= bottom(x_vals[i])

        if j < n - 1:
            A[k, idx(i, j + 1)] = 1
        else:
            b[k] -= top(x_vals[i])

# Метод Гаусса-Зейделя
def gauss_seidel(A, b, eps=1e-4, max_iter=10000):
    n = len(b)
    x = np.zeros(n)
    for k in range(max_iter):
        x_new = np.copy(x)
        for i in range(n):
            s1 = sum(A[i][j] * x_new[j] for j in range(i))
            s2 = sum(A[i][j] * x[j] for j in range(i + 1, n))
            x_new[i] = (b[i] - s1 - s2) / A[i, i]
        if np.linalg.norm(x_new - x, ord=np.inf) < eps:
            return x_new
        x = x_new
    return x

# Решение
solution = gauss_seidel(A, b)

# Вставим результат во внутреннюю сетку
u = np.zeros((n + 1, n + 1))

# Заполнение граничных условий
for j in range(n + 1):
    u[0, j] = left(y_vals[j])
    u[n, j] = right(y_vals[j])
for i in range(n + 1):
    u[i, 0] = bottom(x_vals[i])
    u[i, n] = top(x_vals[i])

# Заполнение внутренних узлов решением
index = 0
for i in range(1, n):
    for j in range(1, n):
        u[i, j] = solution[index]
        index += 1

# Вывод таблицы
u_table = pd.DataFrame(u, index=[f"x={xi:.2f}" for xi in x_vals], columns=[f"y={yj:.2f}" for yj in y_vals])
print("Таблица значений функции u(x, y):")
print(u_table.round(4))


Таблица значений функции u(x, y):
        y=0.00  y=0.25  y=0.50  y=0.75  y=1.00
x=0.00 -1.8415  0.9375  0.7500  0.4375    0.00
x=0.25 -1.6190 -0.2561  0.2088  0.3075    0.25
x=0.50 -1.5323 -0.5516  0.0338  0.3340    0.50
x=0.75 -1.6491 -0.4517  0.1442  0.4945    0.75
x=1.00 -2.0000  0.2500  0.5000  0.7500    1.00
